# ESZA019 — Visão Computacional  
## Laboratório 4 — Item 3D: Correção de distorção de imagens

Este notebook aplica dois métodos do OpenCV para corrigir distorções de lente:

1. `cv2.undistort()`;
2. `cv2.initUndistortRectifyMap()` seguido de `cv2.remap()`.

O procedimento é executado para duas câmeras diferentes, usando os parâmetros intrínsecos e os coeficientes de distorção obtidos nos itens 3B e 3C.

## 1. Organização esperada da pasta

Coloque este notebook na pasta principal do Lab 4, junto com:

```text
parametros_calibracao.npz
cam2_parametros_calibracao.npz
imagem_original_calibracao.jpg
cam2_imagem_colorida.jpg
```

Caso `cam2_imagem_colorida.jpg` não exista, o notebook tentará usar, nesta ordem:

```text
cam2_imagem_original_calibracao.jpg
Capturas_2/cam2_frm0.jpg
```

In [ ]:
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (18, 6)

## 2. Caminhos dos arquivos

In [ ]:
CAM1_PARAMETROS = Path("parametros_calibracao.npz")
CAM2_PARAMETROS = Path("cam2_parametros_calibracao.npz")

CAM1_IMAGENS_CANDIDATAS = [
    Path("imagem_original_calibracao.jpg"),
    Path("Capturas_1/frm0.jpg"),
    Path("frm0.jpg"),
]

CAM2_IMAGENS_CANDIDATAS = [
    Path("cam2_imagem_colorida.jpg"),
    Path("cam2_imagem_original_calibracao.jpg"),
    Path("Capturas_2/cam2_frm0.jpg"),
]


def localizar_primeiro_existente(candidatos):
    for caminho in candidatos:
        if caminho.exists():
            return caminho
    return None


CAM1_IMAGEM = localizar_primeiro_existente(CAM1_IMAGENS_CANDIDATAS)
CAM2_IMAGEM = localizar_primeiro_existente(CAM2_IMAGENS_CANDIDATAS)

print("Parâmetros da câmera 1:", CAM1_PARAMETROS.resolve())
print("Imagem da câmera 1:", CAM1_IMAGEM)
print("Parâmetros da câmera 2:", CAM2_PARAMETROS.resolve())
print("Imagem da câmera 2:", CAM2_IMAGEM)

## 3. Funções auxiliares

A função abaixo:

- carrega a matriz intrínseca \(K\) e o vetor de distorção;
- refina a matriz da câmera com `getOptimalNewCameraMatrix`;
- corrige a imagem por `undistort`;
- cria mapas de correção e aplica `remap`;
- salva os resultados;
- apresenta as imagens lado a lado.

In [ ]:
def carregar_parametros(caminho_npz):
    if not caminho_npz.exists():
        raise FileNotFoundError(
            f"Arquivo de parâmetros não encontrado: {caminho_npz}"
        )

    dados = np.load(caminho_npz, allow_pickle=True)

    chaves = set(dados.files)

    if "camera_matrix" not in chaves:
        raise KeyError(
            f"O arquivo {caminho_npz} não contém 'camera_matrix'. "
            f"Chaves disponíveis: {sorted(chaves)}"
        )

    if "dist_coeffs" not in chaves:
        raise KeyError(
            f"O arquivo {caminho_npz} não contém 'dist_coeffs'. "
            f"Chaves disponíveis: {sorted(chaves)}"
        )

    K = np.asarray(dados["camera_matrix"], dtype=np.float64)
    dist = np.asarray(dados["dist_coeffs"], dtype=np.float64)

    return K, dist


def ler_imagem(caminho):
    if caminho is None:
        raise FileNotFoundError("Nenhuma imagem candidata foi encontrada.")

    imagem = cv2.imread(str(caminho))

    if imagem is None:
        raise ValueError(f"Não foi possível abrir a imagem: {caminho}")

    return imagem


def corrigir_e_exibir(
    nome_camera,
    caminho_parametros,
    caminho_imagem,
    prefixo_saida,
    alpha=1.0,
):
    K, dist = carregar_parametros(caminho_parametros)
    original = ler_imagem(caminho_imagem)

    altura, largura = original.shape[:2]
    tamanho = (largura, altura)

    nova_K, roi = cv2.getOptimalNewCameraMatrix(
        K,
        dist,
        tamanho,
        alpha,
        tamanho,
    )

    # Método 1: cv2.undistort()
    corrigida_undistort = cv2.undistort(
        original,
        K,
        dist,
        None,
        nova_K,
    )

    # Método 2: mapas de correção + cv2.remap()
    mapa_x, mapa_y = cv2.initUndistortRectifyMap(
        K,
        dist,
        None,
        nova_K,
        tamanho,
        cv2.CV_32FC1,
    )

    corrigida_remap = cv2.remap(
        original,
        mapa_x,
        mapa_y,
        interpolation=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
    )

    # Diferença numérica entre as duas implementações
    diferenca = cv2.absdiff(corrigida_undistort, corrigida_remap)
    mae = float(np.mean(diferenca))
    maior_diferenca = int(np.max(diferenca))

    # Salvamento
    cv2.imwrite(f"{prefixo_saida}_original.jpg", original)
    cv2.imwrite(
        f"{prefixo_saida}_corrigida_undistort.jpg",
        corrigida_undistort,
    )
    cv2.imwrite(
        f"{prefixo_saida}_corrigida_remap.jpg",
        corrigida_remap,
    )
    cv2.imwrite(
        f"{prefixo_saida}_diferenca_metodos.jpg",
        diferenca,
    )

    # Recorte opcional da região válida indicada pelo OpenCV
    x, y, w, h = roi

    if w > 0 and h > 0:
        recorte_undistort = corrigida_undistort[y:y+h, x:x+w]
        recorte_remap = corrigida_remap[y:y+h, x:x+w]

        cv2.imwrite(
            f"{prefixo_saida}_corrigida_undistort_recortada.jpg",
            recorte_undistort,
        )
        cv2.imwrite(
            f"{prefixo_saida}_corrigida_remap_recortada.jpg",
            recorte_remap,
        )

    # Conversão apenas para exibição correta no Matplotlib
    imagens_rgb = [
        cv2.cvtColor(original, cv2.COLOR_BGR2RGB),
        cv2.cvtColor(corrigida_undistort, cv2.COLOR_BGR2RGB),
        cv2.cvtColor(corrigida_remap, cv2.COLOR_BGR2RGB),
    ]

    titulos = [
        f"{nome_camera}: original",
        f"{nome_camera}: undistort",
        f"{nome_camera}: remap",
    ]

    figura, eixos = plt.subplots(1, 3, figsize=(18, 6))

    for eixo, imagem_rgb, titulo in zip(eixos, imagens_rgb, titulos):
        eixo.imshow(imagem_rgb)
        eixo.set_title(titulo)
        eixo.axis("off")

    plt.tight_layout()
    plt.show()

    print(f"\n{nome_camera}")
    print("-" * 60)
    print("Imagem:", caminho_imagem)
    print("Resolução:", f"{largura} x {altura}")
    print("Matriz intrínseca K:")
    print(K)
    print("Coeficientes de distorção:")
    print(dist)
    print("Nova matriz da câmera:")
    print(nova_K)
    print("ROI:", roi)
    print("Diferença média absoluta entre os métodos:", mae)
    print("Maior diferença entre os métodos:", maior_diferenca)

    return {
        "K": K,
        "dist": dist,
        "nova_K": nova_K,
        "roi": roi,
        "original": original,
        "undistort": corrigida_undistort,
        "remap": corrigida_remap,
        "mae_metodos": mae,
        "maior_diferenca_metodos": maior_diferenca,
    }

## 4. Correção da imagem da primeira câmera

Os parâmetros utilizados são os obtidos no item 3B.

In [ ]:
resultado_cam1 = corrigir_e_exibir(
    nome_camera="Câmera 1",
    caminho_parametros=CAM1_PARAMETROS,
    caminho_imagem=CAM1_IMAGEM,
    prefixo_saida="cam1_3D",
    alpha=1.0,
)

## 5. Correção da imagem da segunda câmera

Os parâmetros utilizados são os obtidos no item 3C.

In [ ]:
resultado_cam2 = corrigir_e_exibir(
    nome_camera="Câmera 2",
    caminho_parametros=CAM2_PARAMETROS,
    caminho_imagem=CAM2_IMAGEM,
    prefixo_saida="cam2_3D",
    alpha=1.0,
)

## 6. Comparação quantitativa dos métodos

`undistort()` e `remap()` implementam a mesma transformação geométrica. Diferenças pequenas podem surgir devido à construção dos mapas, interpolação e arredondamento numérico.

In [ ]:
print("Câmera 1")
print(
    "Diferença média absoluta:",
    resultado_cam1["mae_metodos"],
)
print(
    "Maior diferença:",
    resultado_cam1["maior_diferenca_metodos"],
)

print("\nCâmera 2")
print(
    "Diferença média absoluta:",
    resultado_cam2["mae_metodos"],
)
print(
    "Maior diferença:",
    resultado_cam2["maior_diferenca_metodos"],
)

## 7. Análise e discussão

Na comparação entre as imagens originais e corrigidas, as alterações devem ser observadas principalmente nas bordas, onde a distorção radial tende a ser mais evidente. Linhas que apareciam levemente curvas na imagem original devem se aproximar de linhas retas após a correção.

Os métodos `cv2.undistort()` e `cv2.remap()` devem produzir resultados visualmente equivalentes, pois ambos utilizam a mesma matriz intrínseca e os mesmos coeficientes de distorção. A diferença é operacional: `undistort()` realiza a correção diretamente, enquanto `remap()` cria previamente mapas de correspondência. A segunda estratégia é especialmente útil quando muitas imagens ou quadros de vídeo da mesma câmera precisam ser corrigidos, porque os mapas podem ser calculados uma única vez e reutilizados.

A matriz obtida por `getOptimalNewCameraMatrix()` controla o compromisso entre preservar o campo de visão e reduzir regiões vazias. Com `alpha = 1`, preserva-se uma parcela maior da imagem original, embora possam aparecer bordas sem informação. Com `alpha = 0`, o OpenCV tende a recortar mais a imagem para eliminar essas regiões.

Para a segunda câmera, os coeficientes radiais estimados foram mais elevados e o erro RMS também foi maior que o obtido para a primeira câmera. Portanto, espera-se que a correção seja mais perceptível na segunda câmera. Caso a imagem corrigida apresente deformações excessivas, isso deve ser discutido como possível consequência da maior incerteza dessa calibração.

## 8. Conclusão

A correção de distorção foi aplicada com sucesso às imagens das duas câmeras por meio de `cv2.undistort()` e da combinação `cv2.initUndistortRectifyMap()`/`cv2.remap()`. Os dois métodos utilizam os parâmetros obtidos durante a calibração e devem apresentar resultados praticamente equivalentes.

A comparação evidencia que os parâmetros de calibração são específicos de cada câmera. Por isso, a matriz intrínseca e os coeficientes de distorção de uma câmera não devem ser utilizados para corrigir imagens capturadas por outro dispositivo.

## 9. Arquivos gerados

Para cada câmera, o notebook salva:

```text
*_original.jpg
*_corrigida_undistort.jpg
*_corrigida_remap.jpg
*_diferenca_metodos.jpg
*_corrigida_undistort_recortada.jpg
*_corrigida_remap_recortada.jpg
```